# AFib detector — external validation on PhysioNet/CinC Challenge 2017 (Notebook 3)
**Bektur Asilov · COMSE-23 · Ala-Too**

Notebooks 1–2 tested the RR-interval TCN only on MIT-BIH AFDB (23 Holter patients from the 1980s).
Here the model is trained **once on all 23 AFDB patients** and then tested, **without any change**, on a different
database: 8,528 short single-lead ECGs (9–61 s, 300 Hz) recorded by patients themselves with the AliveCor KardiaMobile
hand-held device and labelled by the challenge organisers as Normal (N), AF (A), Other rhythm (O) or Too noisy (~).

Rules fixed **before** looking at the test results:
* threshold 0.5, int8 model (as on the device), a decision every 4 beats over the whole recording;
* **record = AF** if at least half of its 32-beat decisions are AF (primary rule); the device-like rule
  "any 2-of-3 alarm" is reported as secondary;
* recordings with fewer than 33 detected beats cannot produce one decision and are reported as *unclassifiable*, not dropped silently;
* main comparison A vs N (the task the model was trained for); how O and ~ are classified is reported separately,
  because a real user also produces those.
* labels: `REFERENCE-v3.csv` (newest relabelled version on PhysioNet).

> Kaggle: *Add Input* → `marufhs/atrial-fibrillation`; Settings → Internet **On** (downloads 95 MB from the PhysioNet open-data mirror). CPU is enough.

In [ ]:
import os, glob, time, random, contextlib, io, json, subprocess, sys, zipfile, urllib.request
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.signal import butter, sosfilt, resample_poly
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'wfdb'], check=True)
import wfdb
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix, roc_auc_score

SEED = 42
FS_ORIG, FS = 250, 125
N_RR, RR_STRIDE, DEV_STRIDE = 32, 8, 4
OUT_DIR = os.environ.get('OUT_DIR', '/kaggle/working')
LABELS = {'(N': 0, '(AFIB': 1}
tf.get_logger().setLevel('ERROR')
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

AFDB_DIR = os.environ.get('AFDB_DIR') or [d for d in {os.path.dirname(p) for p in glob.glob('/kaggle/input/**/04015.dat', recursive=True)} if '/old' not in d][0]
records = sorted(r[:-4] for r in os.listdir(AFDB_DIR) if r.endswith('.dat') and os.path.exists(os.path.join(AFDB_DIR, r[:-4] + '.atr')))
print(len(records), 'AFDB records')

## R-peak detector and RR features (identical to Notebooks 1–2 and the firmware)

In [ ]:
MA = 19; DECAY = 0.5 ** (1 / (2.0 * FS)); THR, REFR, WARM = 0.35, int(0.25 * FS), 2 * FS
try:
    from numba import njit
except ImportError:
    njit = lambda f: f

@njit
def detect_r(x):
    out = np.empty(len(x) // 20 + 10, np.int64); n = 0
    buf = np.zeros(MA); ssum = 0.0; prev = x[0]; pk = 0.0; last = -10**9; armed = True
    for i in range(len(x)):
        d = x[i] - prev; prev = x[i]; e = d * d
        ssum += e - buf[i % MA]; buf[i % MA] = e
        env = ssum / MA; pk = max(pk * DECAY, env)
        if i < WARM: continue
        if armed and env > THR * pk and i - last > REFR:
            out[n] = i; n += 1; last = i; armed = False
            if n >= len(out): break
        elif not armed and env < 0.5 * THR * pk:
            armed = True
    return out[:n]

def rr_features(rr):
    rr = np.clip(rr, 0.2, 2.0)
    return np.stack([(rr - 0.8) / 0.25, (rr / rr.mean() - 1) / 0.15], -1).astype(np.float32)

def to_125(sig, fs):
    # same band-pass as the firmware (0.5-40 Hz, 2nd order), then to 125 Hz
    x = sosfilt(butter(2, [0.5, 40], btype='band', fs=fs, output='sos'), np.nan_to_num(sig))
    if fs == 250: return x[::2]
    from math import gcd
    g = gcd(int(fs), FS); return resample_poly(x, FS // g, int(fs) // g)

## Train the final model on all 23 AFDB patients

In [ ]:
def tcn_block(x, f, k, d, drop):
    sc = x
    x = layers.Conv1D(f, k, padding='causal', dilation_rate=d, activation='relu')(x)
    x = layers.Dropout(drop)(x)
    x = layers.Conv1D(f, k, padding='causal', dilation_rate=d)(x)
    if sc.shape[-1] != f: sc = layers.Conv1D(f, 1)(sc)
    return layers.ReLU()(layers.Add()([x, sc]))

def build_rr_tcn():
    inp = keras.Input(shape=(N_RR, 2)); x = inp
    for d in (1, 2, 4, 8): x = tcn_block(x, 16, 3, d, 0.1)
    return keras.Model(inp, layers.Dense(1, activation='sigmoid')(layers.GlobalAveragePooling1D()(x)))

def to_int8(m, Xrep):
    conv = tf.lite.TFLiteConverter.from_keras_model(m)
    conv.optimizations = [tf.lite.Optimize.DEFAULT]
    rep = Xrep[np.random.RandomState(0).choice(len(Xrep), min(1000, len(Xrep)), replace=False)]
    conv.representative_dataset = lambda: ([s[None]] for s in rep)
    conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    conv.inference_input_type = tf.int8; conv.inference_output_type = tf.int8
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        return conv.convert()

def run_tflite(blob, Xd, bs=4096):
    outs = []
    it = tf.lite.Interpreter(model_content=blob)
    for i in range(0, len(Xd), bs):
        xb = Xd[i:i + bs]; inp = it.get_input_details()[0]
        it.resize_tensor_input(inp['index'], [len(xb), N_RR, 2]); it.allocate_tensors()
        inp, out = it.get_input_details()[0], it.get_output_details()[0]
        sc, zp = inp['quantization']; it.set_tensor(inp['index'], np.clip(np.round(xb / sc + zp), -128, 127).astype(np.int8))
        it.invoke(); sc, zp = out['quantization']
        outs.append(((it.get_tensor(out['index']).astype(np.float32) - zp) * sc).ravel())
    return np.concatenate(outs) if outs else np.zeros(0)

PREP = {}
for rec in records:
    sig = wfdb.rdrecord(os.path.join(AFDB_DIR, rec), channels=[0]).p_signal[:, 0]
    ann = wfdb.rdann(os.path.join(AFDB_DIR, rec), 'atr')
    starts = list(ann.sample); ends = starts[1:] + [len(sig)]
    segs = [(s, e, LABELS.get(a.strip().rstrip('\x00'))) for s, e, a in zip(starts, ends, ann.aux_note)]
    x = to_125(sig, FS_ORIG); beats = detect_r(x); rr = np.diff(beats) / FS
    lab_at = np.full(len(x), -1, np.int8)
    for s, e, l in segs:
        if l is not None: lab_at[s // 2:e // 2] = l
    Xw, yw = [], []
    for k in range(0, len(rr) - N_RR + 1, RR_STRIDE):
        labs = lab_at[beats[k]:beats[k + N_RR]]
        if len(labs) == 0 or labs.min() < 0 or labs.min() != labs.max(): continue
        Xw.append(rr_features(rr[k:k + N_RR])); yw.append(int(labs[0]))
    PREP[rec] = (np.array(Xw, np.float32).reshape(-1, N_RR, 2), np.array(yw, np.int32))

rng = np.random.RandomState(SEED)
for _ in range(500):   # 2 patients only for early stopping, both rhythms present (same rule as Notebook 2)
    val = list(rng.choice(records, 2, replace=False))
    if all(sum((PREP[v][1] == c).sum() for v in val) >= 100 for c in (0, 1)): break
train = [r for r in records if r not in val]
Xtr = np.concatenate([PREP[r][0] for r in train]); ytr = np.concatenate([PREP[r][1] for r in train])
Xva = np.concatenate([PREP[r][0] for r in val]);   yva = np.concatenate([PREP[r][1] for r in val])
cw = {0: len(ytr) / (2 * (ytr == 0).sum()), 1: len(ytr) / (2 * ytr.sum())}
model = build_rr_tcn(); model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='binary_crossentropy')
ds = tf.data.Dataset.from_tensor_slices((Xtr, ytr)).shuffle(50000, seed=SEED).batch(256).prefetch(2)
h = model.fit(ds, validation_data=(Xva, yva), epochs=int(os.environ.get('EPOCHS', 40)), class_weight=cw, verbose=0,
              callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
                         keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2)])
BLOB = to_int8(model, Xtr)
open(f'{OUT_DIR}/afib_rr_tcn_all23_int8.tflite', 'wb').write(BLOB)
print('trained on', len(train), 'patients (+2 for early stopping:', val, '), epochs', len(h.history['loss']),
      '| windows', len(ytr), '| int8 model', len(BLOB), 'bytes')

## Download PhysioNet/CinC 2017 and run the frozen model

In [ ]:
CINC_DIR = os.environ.get('CINC_DIR')
if not CINC_DIR:
    # the PhysioNet open-data mirror on AWS is much faster than physionet.org from Kaggle; same files
    MIRRORS = ['https://physionet-open.s3.amazonaws.com/challenge-2017/1.0.0/',
               'https://physionet.org/files/challenge-2017/1.0.0/']
    os.makedirs('/kaggle/tmp', exist_ok=True)
    for f in ('REFERENCE-v3.csv', 'training2017.zip'):
        dst = f'/kaggle/tmp/{f}'
        for base in MIRRORS:
            try:
                t = time.time(); subprocess.run(['curl', '-fsSL', '--max-time', '1200', '-o', dst, base + f], check=True)
                print(f, os.path.getsize(dst) // 1024, 'KB from', base, f'in {time.time()-t:.0f} s', flush=True); break
            except Exception as e:
                print('mirror failed:', base, e, flush=True)
    zipfile.ZipFile('/kaggle/tmp/training2017.zip').extractall('/kaggle/tmp')
    CINC_DIR = '/kaggle/tmp/training2017'
    REF = '/kaggle/tmp/REFERENCE-v3.csv'
else:
    REF = os.path.join(CINC_DIR, 'REFERENCE-v3.csv')
ref = pd.read_csv(REF, header=None, names=['record', 'label'])
print(len(ref), 'recordings;', ref.label.value_counts().to_dict())

def smooth_2of3(pred):
    p = pred.astype(int); s = p.copy(); s[1:] += p[:-1]; s[2:] += p[:-2]
    return (s >= 2).astype(int)

rows, T0 = [], time.time()
for i, (rec, lab) in enumerate(zip(ref.record, ref.label)):
    r = wfdb.rdrecord(os.path.join(CINC_DIR, rec))
    x = to_125(r.p_signal[:, 0], r.fs); beats = detect_r(x); rr = np.diff(beats) / FS
    X = [rr_features(rr[k:k + N_RR]) for k in range(0, len(rr) - N_RR + 1, DEV_STRIDE)]
    row = dict(record=rec, label=lab, dur_s=len(x) / FS, beats=len(beats), n_dec=len(X))
    if X:
        p = run_tflite(BLOB, np.array(X, np.float32))
        row.update(p_mean=float(p.mean()), frac_af=float((p >= .5).mean()), any_2of3=int(smooth_2of3(p >= .5).max()))
    rows.append(row)
    if (i + 1) % 1000 == 0: print(f'{i+1}/{len(ref)}  [{time.time()-T0:.0f} s]', flush=True)
R = pd.DataFrame(rows); R.to_csv(f'{OUT_DIR}/cinc2017_predictions.csv', index=False)

## Results

In [ ]:
ok = R.n_dec > 0
R['pred_major'] = np.where(ok, (R.frac_af >= 0.5).astype(float), np.nan)
R['pred_any'] = np.where(ok, R.any_2of3.astype(float), np.nan)

def binary(df, col):
    d = df[df.label.isin(['A', 'N'])]; y = (d.label == 'A').astype(int).values; p = d[col].values.astype(int)
    tn, fp, fn, tp = confusion_matrix(y, p, labels=[0, 1]).ravel()
    return dict(n_AF=int(y.sum()), n_N=int((1 - y).sum()), TP=int(tp), FN=int(fn), FP=int(fp), TN=int(tn),
                sensitivity=tp / (tp + fn), specificity=tn / (tn + fp), PPV=tp / (tp + fp) if tp + fp else float('nan'),
                F1_AF=2 * tp / (2 * tp + fp + fn))

C = R[ok]
d = C[C.label.isin(['A', 'N'])]
S = {
  'recordings': int(len(R)), 'labels': R.label.value_counts().to_dict(),
  'unclassifiable_lt33_beats': R[~ok].label.value_counts().to_dict(),
  'A_vs_N_majority_rule': binary(C, 'pred_major'),
  'A_vs_N_any_2of3_rule': binary(C, 'pred_any'),
  'A_vs_N_ROC_AUC_mean_prob': float(roc_auc_score((d.label == 'A').astype(int), d.p_mean)),
  'flagged_AF_share_by_label_majority': C.groupby('label').pred_major.mean().round(4).to_dict(),
  'flagged_AF_share_by_label_any': C.groupby('label').pred_any.mean().round(4).to_dict(),
  'reference_in_domain_LOPO_int8_window': dict(sensitivity=0.9834, specificity=0.9649),
}
json.dump(S, open(f'{OUT_DIR}/cinc2017_summary.json', 'w'), indent=2, default=float)
print(json.dumps(S, indent=2, default=lambda v: round(float(v), 4)))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for lab, c in (('N', '#5B8DEF'), ('A', '#C0362F'), ('O', '#E0A030'), ('~', '#888888')):
    ax[0].hist(C[C.label == lab].frac_af, bins=20, range=(0, 1), alpha=.6, density=True, label=lab, color=c)
ax[0].axvline(.5, c='k', ls='--'); ax[0].set_xlabel('share of 32-beat decisions = AF'); ax[0].set_title('per recording'); ax[0].legend()
sh = C.groupby('label').pred_major.mean() * 100
ax[1].bar(sh.index, sh.values, color=['#C0362F' if l == 'A' else '#888' for l in sh.index]); ax[1].set_ylabel('% flagged AF (majority rule)')
ax[1].set_title('flagged as AF, by true label')
b = pd.cut(d.dur_s, [0, 20, 32, 45, 70]); acc = d.assign(ok=(d.pred_major == (d.label == 'A'))).groupby(b, observed=True).ok.mean() * 100
ax[2].bar(acc.index.astype(str), acc.values, color='#5B8DEF'); ax[2].set_ylabel('% correct (A vs N)'); ax[2].set_xlabel('recording length, s')
ax[2].set_title('accuracy vs length')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/cinc2017_results.png', dpi=150); plt.show()
print('inference', round((time.time() - T0) / 60, 1), 'min')